| Feature           | Comes from         | Calculation                |
| ----------------- | ------------------ | -------------------------- |
| HAG               | Ground surface     | Z - ground elevation       |
| Normal vector     | Neighboring points | PCA on neighborhood        |
| Planarity         | Eigenvalues        | PCA                        |
| Linearity         | Eigenvalues        | PCA                        |
| Sphericity        | Eigenvalues        | PCA                        |
| Surface variation | Eigenvalues        | PCA                        |
| Omnivariance      | Eigenvalues        | PCA                        |
| Eigenentropy      | Eigenvalues        | PCA                        |
| Roughness         | Neighboring points | Distance to best-fit plane |
| Density           | Neighbor search    | Points within radius       |
| Height statistics | Neighboring points | min/max/std of Z           |


In [1]:
#!/usr/bin/env python

############################################################
# Use Numpy to Calculate LAS variables
# Ian Horn
# August 7, 2026
############################################################

import json 
import pdal
import numpy as np
from scipy.spatial import cKDTree


In [2]:
lasfile = "/mnt/d/Data/lidar/N075E299.laz"

pipeline = {
    "pipeline": [
        lasfile
    ]
}

p = pdal.Pipeline(json.dumps(pipeline))

count = p.execute()

arrays = p.arrays

points = arrays[0]

# print(points.dtype)
# print(points.dtype.names)
# print(points.shape)


In [3]:
xyz = np.column_stack([
    points["X"],
    points["Y"],
    points["Z"]
])

# xyz


In [4]:
features = np.column_stack([
    points["X"],
    points["Y"],
    points["Z"],
    points["Intensity"],
    points["Classification"]
])

# features


In [11]:
xyz = np.column_stack([
    points["X"],
    points["Y"],
    points["Z"]
])

tree = cKDTree(xyz)
# tree

print(len(xyz), "points loaded into array")


12294882 points loaded into array


In [6]:
# # example of finding neighbors within a radius of 3.0 units from the second point in the dataset
# point = xyz[1]
# idx = tree.query_ball_point(point, r=3.0)
# neighbors = xyz[idx]
# neighbors


# Compute Geometric Features

In [8]:
def compute_geometric_features(tree, xyz, radius):

    n_points = len(xyz)

    planarity = np.zeros(n_points)
    linearity = np.zeros(n_points)
    sphericity = np.zeros(n_points)
    surface_variation = np.zeros(n_points)
    roughness = np.zeros(n_points)

    normal_x = np.zeros(n_points)
    normal_y = np.zeros(n_points)
    normal_z = np.zeros(n_points)

    for i in range(n_points):

        # Find neighboring point indices
        idx = tree.query_ball_point(xyz[i], r=radius)

        # Skip if there aren't enough neighbors for PCA
        if len(idx) < 5:
            continue

        # Extract neighborhood coordinates
        neighbors = xyz[idx]

        # Center the neighborhood
        centroid = neighbors.mean(axis=0)
        centered = neighbors - centroid

        # Compute covariance matrix
        cov = np.cov(centered.T)

        # Eigen decomposition
        eigvals, eigvecs = np.linalg.eigh(cov)

        # Sort largest -> smallest
        order = np.argsort(eigvals)[::-1]
        eigvals = eigvals[order]
        eigvecs = eigvecs[:, order]

        l1, l2, l3 = eigvals

        # Avoid divide-by-zero
        if l1 <= 0:
            continue

        # Derived features
        linearity[i] = (l1 - l2) / l1
        planarity[i] = (l2 - l3) / l1
        sphericity[i] = l3 / l1
        surface_variation[i] = l3 / (l1 + l2 + l3)

        # Smallest eigenvector is the surface normal
        normal = eigvecs[:, 2]

        normal_x[i] = normal[0]
        normal_y[i] = normal[1]
        normal_z[i] = normal[2]

        # Roughness = standard deviation of distances to the plane
        distances = centered @ normal
        roughness[i] = distances.std()

    return {
        "Planarity": planarity,
        "Linearity": linearity,
        "Sphericity": sphericity,
        "SurfaceVariation": surface_variation,
        "Roughness": roughness,
        "NormalX": normal_x,
        "NormalY": normal_y,
        "NormalZ": normal_z,
    }

In [ ]:
geom = compute_geometric_features(tree, xyz, radius=3.0)
# geom

{'Planarity': array([0.77119126, 0.59858091, 0.85976623, ..., 0.70434574, 0.44473542,
        0.09579851], shape=(12294882,)),
 'Linearity': array([0.22847529, 0.40135219, 0.14021719, ..., 0.29535965, 0.55493171,
        0.72519637], shape=(12294882,)),
 'Sphericity': array([3.33448798e-04, 6.68999227e-05, 1.65768688e-05, ...,
        2.94616309e-04, 3.32875401e-04, 1.79005116e-01], shape=(12294882,)),
 'SurfaceVariation': array([1.88191587e-04, 4.18460669e-05, 8.91325639e-06, ...,
        1.72802080e-04, 2.30299662e-04, 1.23128380e-01], shape=(12294882,)),
 'Roughness': array([0.03028307, 0.014543  , 0.00625197, ..., 0.02630214, 0.03330454,
        0.63302251], shape=(12294882,)),
 'NormalX': array([-0.04603992, -0.0696063 , -0.05905876, ...,  0.07704081,
        -0.01747655, -0.57545125], shape=(12294882,)),
 'NormalY': array([ 0.02148171,  0.03580054,  0.02149272, ..., -0.01198656,
        -0.00492651, -0.01626253], shape=(12294882,)),
 'NormalZ': array([ 0.9987086 ,  0.99693194,  0